# E-commerce Funnel Analysis with pandas

## Objective

This notebook analyzes the e-commerce user funnel from product view to purchase. The goal is to identify where users drop off in the conversion journey and compare funnel performance across traffic source and device segments.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), Path.cwd().parent)
     if (candidate / "data" / "users.csv").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the repository root or notebooks directory.")

DATA_DIR = PROJECT_ROOT / "data"
users = pd.read_csv(DATA_DIR / "users.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
events = pd.read_csv(DATA_DIR / "events.csv")


In [5]:
events["event_date"] = pd.to_datetime(events["event_date"])
orders["order_date"] = pd.to_datetime(orders["order_date"])
users["signup_date"] = pd.to_datetime(users["signup_date"])

print("users shape:", users.shape)
print("orders shape:", orders.shape)
print("events shape:", events.shape)

display(users.head())
display(orders.head())
display(events.head())

users shape: (1000, 4)
orders shape: (1500, 5)
events shape: (5000, 5)


,user_id,signup_date,country,device
0,1,2026-04-13,US,desktop
1,2,2026-06-29,China,mobile
2,3,2026-04-03,Canada,mobile
3,4,2026-01-15,US,desktop
4,5,2026-04-17,Mexico,desktop


,order_id,user_id,order_date,amount,status
0,1,983,2026-04-09,84.66,completed
1,2,729,2026-04-12,0.96,completed
2,3,504,2026-03-22,66.57,completed
3,4,456,2026-03-20,11.68,completed
4,5,45,2026-03-08,7.13,cancelled


,event_id,user_id,event_date,event_type,traffic_source
0,1,4,2026-04-21,view_product,paid_search
1,2,854,2026-03-23,add_to_cart,social
2,3,640,2026-04-04,view_product,social
3,4,986,2026-05-06,purchase,organic
4,5,992,2026-04-03,checkout,paid_search


In [6]:
event_type_summary = (
    events
    .groupby("event_type")
    .agg(
        event_count=("event_type", "count"),
        unique_users=("user_id", "nunique")
    )
    .reset_index()
    .sort_values("event_count", ascending=False)
)

event_type_summary

,event_type,event_count,unique_users
3,view_product,2794,950
0,add_to_cart,1231,702
1,checkout,574,436
2,purchase,401,347


## Funnel Definition

The funnel is defined at the user level. A user is counted in a funnel step if they generated at least one event of that type during the analysis period.

Funnel steps:
1. View product
2. Add to cart
3. Checkout
4. Purchase

Each step counts distinct users, not event rows.

In [7]:
funnel_steps = ["view_product", "add_to_cart", "checkout", "purchase"]

funnel_events = events[events["event_type"].isin(funnel_steps)].copy()

funnel_events.head()

,event_id,user_id,event_date,event_type,traffic_source
0,1,4,2026-04-21,view_product,paid_search
1,2,854,2026-03-23,add_to_cart,social
2,3,640,2026-04-04,view_product,social
3,4,986,2026-05-06,purchase,organic
4,5,992,2026-04-03,checkout,paid_search


In [9]:
overall_funnel = (
    funnel_events
    .groupby("event_type")
    .agg(users=("user_id", "nunique"))
    .reset_index()
)

step_order = pd.DataFrame({
    "event_type": funnel_steps,
    "step_number": range(1, len(funnel_steps) + 1)
})

overall_funnel = (
    step_order
    .merge(overall_funnel, on="event_type", how="left")
    .fillna({"users": 0})
)

overall_funnel["users"] = overall_funnel["users"].astype(int)

overall_funnel

,event_type,step_number,users
0,view_product,1,950
1,add_to_cart,2,702
2,checkout,3,436
3,purchase,4,347


In [10]:
overall_funnel["previous_step_users"] = overall_funnel["users"].shift(1)

overall_funnel["step_conversion_rate"] = (
    overall_funnel["users"] / overall_funnel["previous_step_users"]
)

overall_funnel["overall_conversion_rate"] = (
    overall_funnel["users"] / overall_funnel.loc[0, "users"]
)

overall_funnel["dropoff_rate"] = 1 - overall_funnel["step_conversion_rate"]

overall_funnel

,event_type,step_number,users,previous_step_users,step_conversion_rate,overall_conversion_rate,dropoff_rate
0,view_product,1,950,NaN,NaN,1.000000,NaN
1,add_to_cart,2,702,950.0,0.738947,0.738947,0.261053
2,checkout,3,436,702.0,0.621083,0.458947,0.378917
3,purchase,4,347,436.0,0.795872,0.365263,0.204128


In [11]:
source_funnel = (
    events
    .assign(
        view_user=np.where(events["event_type"] == "view_product", events["user_id"], np.nan),
        cart_user=np.where(events["event_type"] == "add_to_cart", events["user_id"], np.nan),
        checkout_user=np.where(events["event_type"] == "checkout", events["user_id"], np.nan),
        purchase_user=np.where(events["event_type"] == "purchase", events["user_id"], np.nan)
    )
    .groupby("traffic_source")
    .agg(
        view_users=("view_user", "nunique"),
        cart_users=("cart_user", "nunique"),
        checkout_users=("checkout_user", "nunique"),
        purchase_users=("purchase_user", "nunique")
    )
    .reset_index()
)

source_funnel["view_to_cart_rate"] = (
    source_funnel["cart_users"] / source_funnel["view_users"].replace(0, np.nan)
)

source_funnel["cart_to_checkout_rate"] = (
    source_funnel["checkout_users"] / source_funnel["cart_users"].replace(0, np.nan)
)

source_funnel["checkout_to_purchase_rate"] = (
    source_funnel["purchase_users"] / source_funnel["checkout_users"].replace(0, np.nan)
)

source_funnel["view_to_purchase_rate"] = (
    source_funnel["purchase_users"] / source_funnel["view_users"].replace(0, np.nan)
)

source_funnel.sort_values("view_to_purchase_rate", ascending=False)

,traffic_source,view_users,cart_users,checkout_users,purchase_users,view_to_cart_rate,cart_to_checkout_rate,checkout_to_purchase_rate,view_to_purchase_rate
3,social,505,254,141,113,0.502970,0.555118,0.801418,0.223762
0,email,538,256,130,112,0.475836,0.507812,0.861538,0.208178
2,paid_search,485,275,132,89,0.567010,0.480000,0.674242,0.183505
1,organic,498,269,134,73,0.540161,0.498141,0.544776,0.146586


## Funnel by User-Level Device Segment

In this simplified dataset, device is stored as a user-level attribute rather than an event-level attribute. Therefore, this analysis compares funnel performance across users' registered or primary device segments, not the actual device used for each event.

In [12]:
events_with_device = events.merge(
    users[["user_id", "device"]],
    on="user_id",
    how="left"
)

device_funnel = (
    events_with_device
    .assign(
        view_user=np.where(events_with_device["event_type"] == "view_product", events_with_device["user_id"], np.nan),
        cart_user=np.where(events_with_device["event_type"] == "add_to_cart", events_with_device["user_id"], np.nan),
        checkout_user=np.where(events_with_device["event_type"] == "checkout", events_with_device["user_id"], np.nan),
        purchase_user=np.where(events_with_device["event_type"] == "purchase", events_with_device["user_id"], np.nan)
    )
    .groupby("device")
    .agg(
        view_users=("view_user", "nunique"),
        cart_users=("cart_user", "nunique"),
        checkout_users=("checkout_user", "nunique"),
        purchase_users=("purchase_user", "nunique")
    )
    .reset_index()
)

device_funnel["view_to_cart_rate"] = (
    device_funnel["cart_users"] / device_funnel["view_users"].replace(0, np.nan)
)

device_funnel["cart_to_checkout_rate"] = (
    device_funnel["checkout_users"] / device_funnel["cart_users"].replace(0, np.nan)
)

device_funnel["checkout_to_purchase_rate"] = (
    device_funnel["purchase_users"] / device_funnel["checkout_users"].replace(0, np.nan)
)

device_funnel["view_to_purchase_rate"] = (
    device_funnel["purchase_users"] / device_funnel["view_users"].replace(0, np.nan)
)

device_funnel.sort_values("view_to_purchase_rate", ascending=False)

,device,view_users,cart_users,checkout_users,purchase_users,view_to_cart_rate,cart_to_checkout_rate,checkout_to_purchase_rate,view_to_purchase_rate
2,tablet,46,34,17,18,0.739130,0.500000,1.058824,0.391304
1,mobile,567,437,274,207,0.770723,0.627002,0.755474,0.365079
0,desktop,337,231,145,122,0.685460,0.627706,0.841379,0.362018


In [13]:
dropoff_analysis = overall_funnel.loc[overall_funnel["dropoff_rate"].notna()].copy()

biggest_dropoff = dropoff_analysis.sort_values(
    "dropoff_rate", 
    ascending=False
).head(1)

biggest_dropoff

,event_type,step_number,users,previous_step_users,step_conversion_rate,overall_conversion_rate,dropoff_rate
2,checkout,3,436,702.0,0.621083,0.458947,0.378917


In [14]:
biggest_dropoff_step = biggest_dropoff["event_type"].iloc[0]
biggest_dropoff_rate = biggest_dropoff["dropoff_rate"].iloc[0]

print(f"The biggest drop-off occurs before the '{biggest_dropoff_step}' step.")
print(f"Drop-off rate: {biggest_dropoff_rate:.1%}")

The biggest drop-off occurs before the 'checkout' step.
Drop-off rate: 37.9%


## Cumulative Funnel Analysis

In [15]:
# Create user-level funnel flags
user_funnel_flags = (
    events
    .pivot_table(
        index="user_id",
        columns="event_type",
        values="event_id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

for step in funnel_steps:
    if step not in user_funnel_flags.columns:
        user_funnel_flags[step] = 0

user_funnel_flags["has_view_product"] = user_funnel_flags["view_product"] > 0
user_funnel_flags["has_add_to_cart"] = user_funnel_flags["add_to_cart"] > 0
user_funnel_flags["has_checkout"] = user_funnel_flags["checkout"] > 0
user_funnel_flags["has_purchase"] = user_funnel_flags["purchase"] > 0

# Cumulative funnel: each step requires all previous steps
cumulative_funnel = pd.DataFrame({
    "step": ["view_product", "add_to_cart", "checkout", "purchase"],
    "users": [
        user_funnel_flags["has_view_product"].sum(),
        (user_funnel_flags["has_view_product"] & user_funnel_flags["has_add_to_cart"]).sum(),
        (user_funnel_flags["has_view_product"] & user_funnel_flags["has_add_to_cart"] & user_funnel_flags["has_checkout"]).sum(),
        (user_funnel_flags["has_view_product"] & user_funnel_flags["has_add_to_cart"] & user_funnel_flags["has_checkout"] & user_funnel_flags["has_purchase"]).sum()
    ]
})

cumulative_funnel["previous_step_users"] = cumulative_funnel["users"].shift(1)
cumulative_funnel["step_conversion_rate"] = (
    cumulative_funnel["users"] / cumulative_funnel["previous_step_users"]
)
cumulative_funnel["overall_conversion_rate"] = (
    cumulative_funnel["users"] / cumulative_funnel.loc[0, "users"]
)
cumulative_funnel["dropoff_rate"] = 1 - cumulative_funnel["step_conversion_rate"]

cumulative_funnel

,step,users,previous_step_users,step_conversion_rate,overall_conversion_rate,dropoff_rate
0,view_product,950,NaN,NaN,1.000000,NaN
1,add_to_cart,669,950.0,0.704211,0.704211,0.295789
2,checkout,289,669.0,0.431988,0.304211,0.568012
3,purchase,98,289.0,0.339100,0.103158,0.660900


## Cumulative Funnel by Traffic Source

In [16]:
source_user_step_flags = (
    events
    .pivot_table(
        index=["traffic_source", "user_id"],
        columns="event_type",
        values="event_id",
        aggfunc="count",
        fill_value=0
    )
    .reset_index()
)

for step in funnel_steps:
    if step not in source_user_step_flags.columns:
        source_user_step_flags[step] = 0

source_user_step_flags["has_view_product"] = source_user_step_flags["view_product"] > 0
source_user_step_flags["has_add_to_cart"] = source_user_step_flags["add_to_cart"] > 0
source_user_step_flags["has_checkout"] = source_user_step_flags["checkout"] > 0
source_user_step_flags["has_purchase"] = source_user_step_flags["purchase"] > 0

source_cumulative_funnel = (
    source_user_step_flags
    .groupby("traffic_source")
    .apply(lambda df: pd.Series({
        "view_users": df["has_view_product"].sum(),
        "cart_users": (df["has_view_product"] & df["has_add_to_cart"]).sum(),
        "checkout_users": (df["has_view_product"] & df["has_add_to_cart"] & df["has_checkout"]).sum(),
        "purchase_users": (df["has_view_product"] & df["has_add_to_cart"] & df["has_checkout"] & df["has_purchase"]).sum()
    }))
    .reset_index()
)

source_cumulative_funnel["view_to_cart_rate"] = (
    source_cumulative_funnel["cart_users"] / source_cumulative_funnel["view_users"].replace(0, np.nan)
)

source_cumulative_funnel["cart_to_checkout_rate"] = (
    source_cumulative_funnel["checkout_users"] / source_cumulative_funnel["cart_users"].replace(0, np.nan)
)

source_cumulative_funnel["checkout_to_purchase_rate"] = (
    source_cumulative_funnel["purchase_users"] / source_cumulative_funnel["checkout_users"].replace(0, np.nan)
)

source_cumulative_funnel["view_to_purchase_rate"] = (
    source_cumulative_funnel["purchase_users"] / source_cumulative_funnel["view_users"].replace(0, np.nan)
)

source_cumulative_funnel.sort_values("view_to_purchase_rate", ascending=False)

,traffic_source,view_users,cart_users,checkout_users,purchase_users,view_to_cart_rate,cart_to_checkout_rate,checkout_to_purchase_rate,view_to_purchase_rate
3,social,505,127,18,2,0.251485,0.141732,0.111111,0.003960
1,organic,498,132,14,1,0.265060,0.106061,0.071429,0.002008
0,email,538,137,13,1,0.254647,0.094891,0.076923,0.001859
2,paid_search,485,135,17,0,0.278351,0.125926,0.000000,0.000000


## Cumulative Funnel by User-Level Device Segment

Because device is stored at the user level in this dataset, this analysis compares cumulative funnel performance across users' registered or primary device segment.

In [17]:
user_funnel_with_device = user_funnel_flags.merge(
    users[["user_id", "device"]],
    on="user_id",
    how="left"
)

device_cumulative_funnel = (
    user_funnel_with_device
    .groupby("device")
    .apply(lambda df: pd.Series({
        "view_users": df["has_view_product"].sum(),
        "cart_users": (df["has_view_product"] & df["has_add_to_cart"]).sum(),
        "checkout_users": (df["has_view_product"] & df["has_add_to_cart"] & df["has_checkout"]).sum(),
        "purchase_users": (df["has_view_product"] & df["has_add_to_cart"] & df["has_checkout"] & df["has_purchase"]).sum()
    }))
    .reset_index()
)

device_cumulative_funnel["view_to_cart_rate"] = (
    device_cumulative_funnel["cart_users"] / device_cumulative_funnel["view_users"].replace(0, np.nan)
)

device_cumulative_funnel["cart_to_checkout_rate"] = (
    device_cumulative_funnel["checkout_users"] / device_cumulative_funnel["cart_users"].replace(0, np.nan)
)

device_cumulative_funnel["checkout_to_purchase_rate"] = (
    device_cumulative_funnel["purchase_users"] / device_cumulative_funnel["checkout_users"].replace(0, np.nan)
)

device_cumulative_funnel["view_to_purchase_rate"] = (
    device_cumulative_funnel["purchase_users"] / device_cumulative_funnel["view_users"].replace(0, np.nan)
)

device_cumulative_funnel.sort_values("view_to_purchase_rate", ascending=False)

,device,view_users,cart_users,checkout_users,purchase_users,view_to_cart_rate,cart_to_checkout_rate,checkout_to_purchase_rate,view_to_purchase_rate
1,mobile,567,421,192,68,0.742504,0.456057,0.354167,0.119929
2,tablet,46,33,11,4,0.717391,0.333333,0.363636,0.086957
0,desktop,337,215,86,26,0.637982,0.400000,0.302326,0.077151


## Key Findings

1. Under the cumulative user-level funnel definition, 950 users viewed a product, 669 users added to cart after viewing, 289 users reached checkout after adding to cart, and 98 users completed the full purchase journey. This gives a strict end-to-end view-to-purchase conversion rate of 10.3%.

2. The largest drop-off occurs between checkout and purchase, where only 33.9% of users who completed the previous steps eventually purchased. This suggests that the final checkout completion stage may be the biggest friction point in the full user journey.

3. The add-to-cart to checkout stage also shows substantial friction, with a step conversion rate of 43.2% and a drop-off rate of 56.8%. This indicates that both cart-to-checkout entry and final checkout completion should be prioritized for further product investigation.

4. In the cumulative device-level funnel, mobile users show the strongest full-funnel performance, with a view-to-purchase conversion rate of 12.0%, compared with 8.7% for tablet users and 7.7% for desktop users. However, this should be interpreted as user-level device segment performance because device is stored as a user attribute rather than an event-level device field.

5. The non-cumulative funnel produced a much higher view-to-purchase rate of 36.5%, because it counts users independently at each event step. The earlier device-level analysis even produced a checkout-to-purchase rate above 100% for tablet users, revealing that independent step counts can overstate true funnel progression. For portfolio reporting, the cumulative user-level funnel is more appropriate because each step is constrained to users who completed all previous steps.